# Allstate Claims Severity TC1 Human Baseline

I used this notebook to record my simple TC1 from-scratch human baseline for the `allstate-claims-severity` competition.
Each selected preprocessing section is tagged with its `CA-XXXXXX` action ID from `candidate_actions.json`.
This is intentionally a first-pass baseline, not the full curated answer key.


## Load Data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path.home() / 'Downloads' / 'allstate-claims-severity'

required_files = ['train.csv', 'test.csv']
missing = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files under {DATA_DIR}: {missing}')

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

CAT_COLS  = [c for c in train.columns if c.startswith('cat')]
CONT_COLS = [c for c in train.columns if c.startswith('cont')]

print('train shape:', train.shape)
print('test shape: ', test.shape)
print(f'cat cols: {len(CAT_COLS)}, cont cols: {len(CONT_COLS)}')
train.head(2)

## CA-000001 — Drop id Column

`id` is a unique integer row identifier with no relationship to claim severity
Store test IDs separately for submission.

In [ ]:
# CA-000001: DROP_COLUMNS (id)
test_ids = test['id'].copy()

train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

print('train shape after drop:', train.shape)
print('test shape after drop: ', test.shape)

## CA-000005 — Label Encode Categorical Columns

All 116 `cat*` columns hold string values that must be converted to integers
for tree-based models.

`LabelEncoder` is fit on the union of train and test values to ensure
consistent integer codes across both splits.

*Alternative*: one-hot encoding in (CA-000014).

In [ ]:
# CA-000005: ENCODE_CATEGORICAL (label, cat1-cat116, fit_scope=full_data)
for col in CAT_COLS:
    lbl = LabelEncoder()
    lbl.fit(list(train[col].values) + list(test[col].values))
    train[col] = lbl.transform(list(train[col].values))
    test[col]  = lbl.transform(list(test[col].values))

## Action ID Summary

| Step | CA-ID | Action |
|------|-------|--------|
| 1 | CA-000001 | Drop `id` column |
| 2 | CA-000005 | Label encode `cat1`-`cat116` (fit on train+test) |

I stopped before adding the target log transform in this simple human baseline.
The curated bank still contains target-transform alternatives, but this notebook records my intentionally incomplete first-pass selection.
